In [1]:
!pip install hmmlearn

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from hmmlearn import hmm
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Optimized for Google Colab - faster training
print("="*60)
print("GOOGLE COLAB OPTIMIZED BOLUS PREDICTION")
print("="*60)

GOOGLE COLAB OPTIMIZED BOLUS PREDICTION


In [4]:
# Load all patient data (optimized loading)
def load_patient_data_fast(patient_ids):
    all_data = []
    successful_loads = 0

    for pid in patient_ids:
        try:
            df = pd.read_csv(f'{pid}.csv', delimiter=';', usecols=['time', 'glucose', 'calories', 'heart_rate', 'steps', 'basal_rate', 'bolus_volume_delivered', 'carb_input'])
            df['patient_id'] = pid
            all_data.append(df)
            successful_loads += 1
            if successful_loads % 5 == 0:
                print(f"Loaded {successful_loads} patients...")
        except:
            continue

    print(f"\n✓ Successfully loaded {successful_loads} patients")
    return pd.concat(all_data, ignore_index=True)

In [5]:
# Define patient IDs
patient_ids = []
for i in range(1, 26):
    if i not in [8, 12, 13]:
        patient_ids.append(f'HUPA{i:04d}P')

for i in range(26, 29):
    patient_ids.append(f'HUPA{i:04d}P')

print(f"\nLoading {len(patient_ids)} patients...")
df = load_patient_data_fast(patient_ids)


Loading 25 patients...
Loaded 5 patients...
Loaded 10 patients...
Loaded 15 patients...
Loaded 20 patients...
Loaded 25 patients...

✓ Successfully loaded 25 patients


In [6]:
# Quick data preparation
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values(['patient_id', 'time'])

print(f"\nData Statistics:")
print(f"Total records: {len(df):,}")
print(f"Patients: {df['patient_id'].nunique()}")
print(f"Time range: {df['time'].min()} to {df['time'].max()}")


Data Statistics:
Total records: 309,392
Patients: 25
Time range: 2018-06-13 18:40:00 to 2022-05-18 12:15:00


In [7]:
# Create sequences (optimized)
def create_sequences_fast(data, sequence_length=6, target_col='bolus_volume_delivered'):
    """Faster sequence creation with shorter sequences"""
    sequences = []
    targets = []

    feature_cols = ['glucose', 'calories', 'heart_rate', 'steps', 'basal_rate', 'carb_input']

    for patient in data['patient_id'].unique():
        patient_data = data[data['patient_id'] == patient]

        if len(patient_data) <= sequence_length:
            continue

        # Simple normalization (min-max per feature)
        for col in feature_cols:
            min_val = patient_data[col].min()
            max_val = patient_data[col].max()
            if max_val > min_val:
                patient_data[col] = (patient_data[col] - min_val) / (max_val - min_val)

        features = patient_data[feature_cols].values

        for i in range(len(patient_data) - sequence_length):
            seq = features[i:i+sequence_length]
            target = patient_data[target_col].iloc[i+sequence_length]
            sequences.append(seq)
            targets.append(target)

    print(f"Created {len(sequences)} sequences")
    return np.array(sequences), np.array(targets)

print("\nCreating sequences (6 timesteps = 30 minutes)...")
X, y = create_sequences_fast(df, sequence_length=6)
print(f"X shape: {X.shape}, y shape: {y.shape}")


Creating sequences (6 timesteps = 30 minutes)...
Created 309242 sequences
X shape: (309242, 6, 6), y shape: (309242,)


In [9]:
# Split data (simple split for speed)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)
print(f"\nTrain set: {X_train.shape}")
print(f"Test set: {X_test.shape}")


Train set: (247393, 6, 6)
Test set: (61849, 6, 6)


In [10]:
# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train).view(-1, 1)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test).view(-1, 1)

In [11]:
# Lightweight LSTM Model
class LightLSTM(nn.Module):
    def __init__(self, input_size=6, hidden_size=32, output_size=1):
        super(LightLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        lstm_out = lstm_out[:, -1, :]
        output = self.fc(lstm_out)
        return output

# Lightweight GRU Model
class LightGRU(nn.Module):
    def __init__(self, input_size=6, hidden_size=32, output_size=1):
        super(LightGRU, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        gru_out, _ = self.gru(x)
        gru_out = gru_out[:, -1, :]
        output = self.fc(gru_out)
        return output

In [12]:
# Fast training function
def train_fast(model, X_train, y_train, X_test, y_test, epochs=20, lr=0.001, model_name="Model"):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training on {device}")
    model = model.to(device)

    # Create data loaders
    train_dataset = TensorDataset(X_train.to(device), y_train.to(device))
    test_dataset = TensorDataset(X_test.to(device), y_test.to(device))

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Train
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Quick validation every 5 epochs
        if (epoch + 1) % 5 == 0 or epoch == epochs - 1:
            model.eval()
            with torch.no_grad():
                test_outputs = model(X_test.to(device))
                test_loss = criterion(test_outputs, y_test.to(device))

            print(f"{model_name} - Epoch {epoch+1}/{epochs}: "
                  f"Train Loss: {train_loss/len(train_loader):.4f}, "
                  f"Test Loss: {test_loss.item():.4f}")

    return model

print("\n" + "="*60)
print("TRAINING LIGHTWEIGHT MODELS (20 EPOCHS)")
print("="*60)


TRAINING LIGHTWEIGHT MODELS (20 EPOCHS)


In [13]:
# Initialize and train models
lstm_model = LightLSTM(input_size=6, hidden_size=32)
gru_model = LightGRU(input_size=6, hidden_size=32)

print("\nTraining LSTM...")
lstm_model = train_fast(lstm_model, X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor,
                        epochs=20, lr=0.001, model_name="LSTM")

print("\nTraining GRU...")
gru_model = train_fast(gru_model, X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor,
                       epochs=20, lr=0.001, model_name="GRU")


Training LSTM...
Training on cpu
LSTM - Epoch 5/20: Train Loss: 0.5631, Test Loss: 0.5787
LSTM - Epoch 10/20: Train Loss: 0.5617, Test Loss: 0.5771
LSTM - Epoch 15/20: Train Loss: 0.5600, Test Loss: 0.5763
LSTM - Epoch 20/20: Train Loss: 0.5582, Test Loss: 0.5775

Training GRU...
Training on cpu
GRU - Epoch 5/20: Train Loss: 0.5636, Test Loss: 0.5788
GRU - Epoch 10/20: Train Loss: 0.5620, Test Loss: 0.5773
GRU - Epoch 15/20: Train Loss: 0.5607, Test Loss: 0.5767
GRU - Epoch 20/20: Train Loss: 0.5592, Test Loss: 0.5775


In [14]:
# Simple HMM (fast training)
print("\n" + "="*60)
print("TRAINING SIMPLE HMM")
print("="*60)

# Prepare HMM data (discretize bolus into 5 states)
bolus_values = np.unique(y)
if len(bolus_values) > 5:
    # Reduce to 5 states for faster HMM
    y_disc = np.digitize(y, np.percentile(y, [0, 20, 40, 60, 80, 100])) - 1
    n_states = 5
else:
    y_disc = y
    n_states = len(bolus_values)


TRAINING SIMPLE HMM


In [15]:
# Use glucose and heart rate as observations
obs_features = np.column_stack([X[:, -1, 0], X[:, -1, 2]])  # Last timestep glucose and heart rate

print(f"HMM States: {n_states}")
print(f"Observations shape: {obs_features.shape}")

HMM States: 5
Observations shape: (309242, 2)


In [16]:
# Train HMM
hmm_model = hmm.GaussianHMM(
    n_components=n_states,
    covariance_type="diag",
    n_iter=50,  # Fewer iterations
    random_state=42
)

hmm_model.fit(obs_features)
print("✓ HMM trained in 50 iterations")

✓ HMM trained in 50 iterations


In [17]:
# Evaluation function
def evaluate_fast(model, X_test, y_test, model_name="Model"):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()

    with torch.no_grad():
        predictions = model(X_test.to(device)).cpu().numpy().flatten()

    actuals = y_test.numpy().flatten()

    # Calculate metrics
    mae = np.mean(np.abs(predictions - actuals))
    rmse = np.sqrt(np.mean((predictions - actuals) ** 2))

    # For classification (since bolus is often 0, 0.5, 1.0, etc.)
    predicted_classes = np.round(predictions * 2) / 2
    actual_classes = np.round(actuals * 2) / 2
    accuracy = np.mean(predicted_classes == actual_classes)

    print(f"\n{model_name} Results:")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Accuracy (rounded to nearest 0.5): {accuracy:.4f}")

    # Show top 5 predictions vs actual
    print("\nSample predictions vs actual:")
    for i in range(min(5, len(predictions))):
        print(f"  Pred: {predictions[i]:.2f} | Actual: {actuals[i]:.2f}")

    return predictions, actuals, {'MAE': mae, 'RMSE': rmse, 'Accuracy': accuracy}

In [20]:
# Fix for evaluate_hmm_fast function
def evaluate_hmm_fast(hmm_model, X_test, y_test, n_states=5):
    # Use last timestep glucose and heart rate
    obs_test = np.column_stack([X_test[:, -1, 0], X_test[:, -1, 2]])

    # Predict states
    predicted_states = hmm_model.predict(obs_test)

    # Map states to bolus values (simple mapping)
    # Convert y_test to numpy if it's a tensor
    if hasattr(y_test, 'numpy'):
        actuals = y_test.numpy().flatten()
    else:
        actuals = y_test.flatten()

    # For simplicity, use mean bolus per state from training
    state_means = []
    for state in range(n_states):
        state_mask = (hmm_model.predict(obs_features) == state)
        if np.any(state_mask):
            state_means.append(np.mean(y[state_mask]))
        else:
            state_means.append(0)

    predictions = np.array([state_means[state] for state in predicted_states])

    # Calculate metrics
    mae = np.mean(np.abs(predictions - actuals))
    rmse = np.sqrt(np.mean((predictions - actuals) ** 2))

    predicted_classes = np.round(predictions * 2) / 2
    actual_classes = np.round(actuals * 2) / 2
    accuracy = np.mean(predicted_classes == actual_classes)

    print(f"\nHMM Results:")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Accuracy: {accuracy:.4f}")

    print("\nSample predictions vs actual:")
    for i in range(min(5, len(predictions))):
        print(f"  Pred: {predictions[i]:.2f} | Actual: {actuals[i]:.2f}")

    return predictions, actuals, {'MAE': mae, 'RMSE': rmse, 'Accuracy': accuracy}

In [21]:
# Evaluate all models
print("\nEvaluating LSTM...")
lstm_preds, lstm_actuals, lstm_metrics = evaluate_fast(lstm_model, X_test_tensor, y_test_tensor, "LSTM")

print("\nEvaluating GRU...")
gru_preds, gru_actuals, gru_metrics = evaluate_fast(gru_model, X_test_tensor, y_test_tensor, "GRU")

print("\nEvaluating HMM...")
hmm_preds, hmm_actuals, hmm_metrics = evaluate_hmm_fast(hmm_model, X_test, y_test, n_states)

# Save models (lightweight versions)
print("\n" + "="*60)
print("SAVING MODELS")
print("="*60)

torch.save(lstm_model.state_dict(), 'lstm_light.pth')
torch.save(gru_model.state_dict(), 'gru_light.pth')


Evaluating LSTM...

LSTM Results:
MAE: 0.1341
RMSE: 0.7599
Accuracy (rounded to nearest 0.5): 0.9827

Sample predictions vs actual:
  Pred: 0.06 | Actual: 0.00
  Pred: 0.05 | Actual: 0.00
  Pred: 0.09 | Actual: 0.00
  Pred: 0.19 | Actual: 0.00
  Pred: 0.02 | Actual: 0.00

Evaluating GRU...

GRU Results:
MAE: 0.1461
RMSE: 0.7600
Accuracy (rounded to nearest 0.5): 0.9689

Sample predictions vs actual:
  Pred: 0.02 | Actual: 0.00
  Pred: 0.09 | Actual: 0.00
  Pred: 0.04 | Actual: 0.00
  Pred: 0.19 | Actual: 0.00
  Pred: -0.02 | Actual: 0.00

Evaluating HMM...

HMM Results:
MAE: 0.1372
RMSE: 0.7635
Accuracy: 0.9880

Sample predictions vs actual:
  Pred: 0.08 | Actual: 0.00
  Pred: 0.08 | Actual: 0.00
  Pred: 0.08 | Actual: 0.00
  Pred: 0.08 | Actual: 0.00
  Pred: 0.08 | Actual: 0.00

SAVING MODELS


In [22]:
import joblib
joblib.dump(hmm_model, 'hmm_light.pkl')

# Save model info
model_info = {
    'sequence_length': 6,
    'features': ['glucose', 'calories', 'heart_rate', 'steps', 'basal_rate', 'carb_input'],
    'input_size': 6,
    'models_saved': ['lstm_light.pth', 'gru_light.pth', 'hmm_light.pkl']
}
joblib.dump(model_info, 'model_info.pkl')

['model_info.pkl']

In [23]:
print("✓ Models saved successfully!")
print("  - LSTM: lstm_light.pth")
print("  - GRU: gru_light.pth")
print("  - HMM: hmm_light.pkl")
print("  - Info: model_info.pkl")

# Comparison table
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)

comparison = pd.DataFrame({
    'Model': ['LSTM', 'GRU', 'HMM'],
    'MAE': [lstm_metrics['MAE'], gru_metrics['MAE'], hmm_metrics['MAE']],
    'RMSE': [lstm_metrics['RMSE'], gru_metrics['RMSE'], hmm_metrics['RMSE']],
    'Accuracy': [lstm_metrics['Accuracy'], gru_metrics['Accuracy'], hmm_metrics['Accuracy']]
})

print(comparison.to_string(index=False))

✓ Models saved successfully!
  - LSTM: lstm_light.pth
  - GRU: gru_light.pth
  - HMM: hmm_light.pkl
  - Info: model_info.pkl

MODEL COMPARISON
Model      MAE     RMSE  Accuracy
 LSTM 0.134119 0.759912  0.982651
  GRU 0.146113 0.759965  0.968908
  HMM 0.137231 0.763452  0.988035


In [24]:
# Simple prediction function for deployment
def predict_bolus_light(recent_data, model_type='lstm'):
    """
    Fast prediction function for Google Colab

    Parameters:
    recent_data: numpy array of shape (6, 6) - 6 timesteps, 6 features
    model_type: 'lstm', 'gru', or 'hmm'
    """

    if recent_data.shape != (6, 6):
        raise ValueError(f"Expected shape (6, 6), got {recent_data.shape}")

    if model_type == 'lstm':
        # Load and predict with LSTM
        model = LightLSTM(input_size=6, hidden_size=32)
        model.load_state_dict(torch.load('lstm_light.pth', map_location='cpu'))
        model.eval()

        input_tensor = torch.FloatTensor(recent_data).unsqueeze(0)
        with torch.no_grad():
            prediction = model(input_tensor)

        return prediction.item()

    elif model_type == 'gru':
        # Load and predict with GRU
        model = LightGRU(input_size=6, hidden_size=32)
        model.load_state_dict(torch.load('gru_light.pth', map_location='cpu'))
        model.eval()

        input_tensor = torch.FloatTensor(recent_data).unsqueeze(0)
        with torch.no_grad():
            prediction = model(input_tensor)

        return prediction.item()

    elif model_type == 'hmm':
        # Load and predict with HMM
        hmm_model = joblib.load('hmm_light.pkl')

        # Use last glucose and heart rate
        obs = np.array([[recent_data[-1, 0], recent_data[-1, 2]]])
        state = hmm_model.predict(obs)[0]

        # Simple mapping (in practice, you'd want a better mapping)
        state_to_bolus = {0: 0.0, 1: 0.5, 2: 1.0, 3: 2.0, 4: 3.0}
        return state_to_bolus.get(state, 0.0)

    else:
        raise ValueError("model_type must be 'lstm', 'gru', or 'hmm'")

In [25]:
# Example prediction
print("\n" + "="*60)
print("EXAMPLE PREDICTION")
print("="*60)

# Create sample data (last 6 timesteps of first test sample)
sample_data = X_test[0]
actual_value = y_test[0].item()

print(f"Sample data shape: {sample_data.shape}")
print(f"Actual bolus: {actual_value:.2f}")


EXAMPLE PREDICTION
Sample data shape: (6, 6)
Actual bolus: 0.00


In [27]:
try:
    lstm_pred = predict_bolus_light(sample_data, 'lstm')
    gru_pred = predict_bolus_light(sample_data, 'gru')
    hmm_pred = predict_bolus_light(sample_data, 'hmm')

    print(f"\nLSTM prediction: {lstm_pred:.2f}")
    print(f"GRU prediction: {gru_pred:.2f}")
    print(f"HMM prediction: {hmm_pred:.2f}")

    print(f"\nErrors:")
    print(f"LSTM error: {abs(lstm_pred - actual_value):.2f}")
    print(f"GRU error: {abs(gru_pred - actual_value):.2f}")
    print(f"HMM error: {abs(hmm_pred - actual_value):.2f}")
except Exception as e:
    print(f"Prediction error: {e}")

print("\n" + "="*60)
print("PERFORMANCE SUMMARY")
print("="*60)
print(f"Total training time estimate: ~2-3 minutes on Google Colab")
print(f"Sequence length: 6 timesteps (30 minutes)")
print(f"Training epochs: 20")
print(f"Batch size: 64")
print(f"Models: Lightweight LSTM, GRU, and HMM")
print(f"Memory usage: Low")
print(f"Models saved for immediate use")


LSTM prediction: 0.06
GRU prediction: 0.02
HMM prediction: 2.00

Errors:
LSTM error: 0.06
GRU error: 0.02
HMM error: 2.00

PERFORMANCE SUMMARY
Total training time estimate: ~2-3 minutes on Google Colab
Sequence length: 6 timesteps (30 minutes)
Training epochs: 20
Batch size: 64
Models: Lightweight LSTM, GRU, and HMM
Memory usage: Low
Models saved for immediate use
